In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# supplier_master already loaded
df_suppliers = pd.read_csv("data/raw/df_suppliers.csv")

n_po = 10000

component_map = {
    "electronics": ["CMP_ELEC_001", "CMP_ELEC_002"],
    "metals": ["CMP_MET_001", "CMP_MET_002"],
    "packaging": ["CMP_PKG_001"],
    "chemicals": ["CMP_CHEM_001"],
    "plastics": ["CMP_PLAS_001"],
    "logistics": ["CMP_LOG_001"],
    "textiles": ["CMP_TEXT_001"]
}

unit_prices = {
    "CMP_ELEC_001": 45,
    "CMP_ELEC_002": 120,
    "CMP_MET_001": 15,
    "CMP_MET_002": 22,
    "CMP_PKG_001": 2,
    "CMP_CHEM_001": 8,
    "CMP_PLAS_001": 4,
    "CMP_LOG_001": 1,
    "CMP_TEXT_001": 6
}

rows = []

supplier_weights = (
    df_suppliers["contract_value"] /
    df_suppliers["contract_value"].sum()
)

for i in range(n_po):

    supplier = df_suppliers.sample(
        1,
        weights=supplier_weights
    ).iloc[0]

    category = supplier["supplier_category"]

    component = random.choice(
        component_map[category]
    )

    order_date = pd.Timestamp("2025-01-01") + pd.to_timedelta(
        np.random.randint(0, 365),
        unit="D"
    )

    lead_time = {
        "electronics": np.random.randint(30, 91),
        "metals": np.random.randint(15, 46),
        "packaging": np.random.randint(7, 21)
    }.get(category, np.random.randint(10, 40))

    expected_delivery = order_date + timedelta(days=lead_time)

    delay = np.random.choice(
        [-3,-1,0,0,0,2,5,10,20],
        p=[0.03,0.07,0.4,0.2,0.1,0.1,0.05,0.03,0.02]
    )

    actual_delivery = expected_delivery + timedelta(days=int(delay))

    quantity = max(
        1,
        int(np.random.lognormal(mean=6, sigma=1))
    )

    received = quantity

    partial_prob = np.random.rand()

    if partial_prob < 0.1:
        received = int(quantity * np.random.uniform(0.7,0.98))

    unit_price = unit_prices[component]

    order_value = round(quantity * unit_price, 2)

    rows.append({
        "po_id": f"PO{i+1:07}",
        "supplier_id": supplier["supplier_id"],
        "order_date": order_date.date(),
        "expected_delivery_date": expected_delivery.date(),
        "actual_delivery_date": actual_delivery.date(),
        "order_quantity": quantity,
        "received_quantity": received,
        "order_value": order_value,
        "component_id": component
    })

df_po = pd.DataFrame(rows)

In [ ]:
df_po.head()

In [ ]:
df_po.info()

In [ ]:
df_po.to_csv('data/raw/df_po.csv', index=False)